# 01 - First look: Open Psychometrics RIASEC dataset (12 Dec 2018)
Read-only exploration. The original files are never modified and nothing is written to `processed/` yet.

## 1. Inventory
List files, then **detect the delimiter** by counting candidate separators in the header line (don't assume comma).

In [1]:
from pathlib import Path
import re, pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_rows", 200)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "RIASEC_data12Dec2018"
for p in sorted(DATA.rglob("*")):
    if p.is_file(): print(f"{p.stat().st_size:>12,}  {p.relative_to(ROOT)}")

          71  RIASEC_data12Dec2018\.~lock.data.csv#
       6,088  RIASEC_data12Dec2018\codebook.txt
  29,324,901  RIASEC_data12Dec2018\data.csv


In [2]:
header = open(DATA / "data.csv", encoding="utf-8", errors="replace").readline()
counts = {d: header.count(d) for d in [",", "\t", ";", "|"]}
delim = max(counts, key=counts.get)
print("separator counts in header:", counts, "->", repr(delim))

# dtype=str + keep_default_na=False: read everything as raw text so that blanks and
# strings like "NA" are NOT silently turned into NaN. We convert numeric columns ourselves.
raw = pd.read_csv(DATA / "data.csv", sep=delim, dtype=str, keep_default_na=False, encoding_errors="replace")
print("shape:", raw.shape)
stray = raw.columns[-1]
print("last column name:", repr(stray), "| non-blank rows in it:", (raw[stray].str.strip() != "").sum())

separator counts in header: {',': 0, '\t': 93, ';': 0, '|': 0} -> '\t'
shape: (145828, 94)
last column name: 'Unnamed: 93' | non-blank rows in it: 1


### Column groups (from the codebook)

In [3]:
groups = {
 "RIASEC items (48; R1-R8, I1-I8, A1-A8, S1-S8, E1-E8, C1-C8), 1-5 dislike..enjoy": [f"{l}{i}" for l in "RIASEC" for i in range(1, 9)],
 "timing (seconds)": ["introelapse", "testelapse", "surveyelapse"],
 "TIPI Big-Five items (10), 1-7": [f"TIPI{i}" for i in range(1, 11)],
 "VCL vocabulary checklist (16; VCL6, VCL9, VCL12 are fake words)": [f"VCL{i}" for i in range(1, 17)],
 "demographics": ["education","urban","gender","engnat","age","hand","religion","orientation","race","voted","married","familysize"],
 "technical": ["uniqueNetworkLocation", "country", "source"],
 "field of study": ["major"],
}
for g, cols in groups.items():
    missing = [c for c in cols if c not in raw.columns]
    print(f"{len(cols):>3} cols  {g}   (absent from file: {missing or 'none'})")

 48 cols  RIASEC items (48; R1-R8, I1-I8, A1-A8, S1-S8, E1-E8, C1-C8), 1-5 dislike..enjoy   (absent from file: none)
  3 cols  timing (seconds)   (absent from file: none)
 10 cols  TIPI Big-Five items (10), 1-7   (absent from file: none)
 16 cols  VCL vocabulary checklist (16; VCL6, VCL9, VCL12 are fake words)   (absent from file: none)
 12 cols  demographics   (absent from file: none)
  3 cols  technical   (absent from file: none)
  1 cols  field of study   (absent from file: none)


## 2. License / terms
Search every text file for license, permission or citation wording.

In [4]:
pat = re.compile(r"licen[sc]e|cite|citation|copyright|permission|creative commons|terms of use|reference", re.I)
for p in DATA.rglob("*"):
    if p.is_file() and p.suffix.lower() in {".txt", ".md", ".html", ".htm"} and p.stat().st_size < 1_000_000:
        for n, line in enumerate(open(p, encoding="utf-8", errors="replace"), 1):
            if pat.search(line): print(p.name, n, line.strip()[:300])
print("(end of search)")
# The only citation-like text in the codebook is for the TIPI questionnaire, not the dataset.

(end of search)


## 3. The `major` column

In [5]:
major = raw["major"]
blank = major.str.strip() == ""
n = len(raw)
print(f"column exists: {'major' in raw.columns}")
print(f"filled (non-blank): {(~blank).sum():>8,}  ({(~blank).mean():.1%})")
print(f"empty / blank     : {blank.sum():>8,}  ({blank.mean():.1%})")
print("   of which empty string:", (major == "").sum(), "| whitespace only:", ((major != "") & blank).sum())
junk = major.str.strip().str.lower().isin(["na", "n/a", "none", "no", "nothing", "-", ".", "x", "idk", "?"])
print(f"filled but placeholder-like (NA/none/no/-/idk...): {junk.sum():,}")
print("distinct raw values:", major[~blank].nunique(), "| distinct after strip+lowercase:", major[~blank].str.strip().str.lower().nunique())

column exists: True
filled (non-blank):   93,626  (64.2%)
empty / blank     :   52,202  (35.8%)
   of which empty string: 52193 | whitespace only: 9
filled but placeholder-like (NA/none/no/-/idk...): 1,752
distinct raw values: 15957 | distinct after strip+lowercase: 11816


In [6]:
top = major[~blank].value_counts().head(50)
out = pd.DataFrame({"raw value": [repr(v) for v in top.index], "count": top.values, "% of filled": (top.values / (~blank).sum() * 100).round(2)})
out.index = range(1, 51)
print(out.to_string())

                    raw value  count  % of filled
1                'psychology'   6861         7.33
2                'Psychology'   5763         6.16
3                   'English'   2342         2.50
4                  'Business'   2290         2.45
5                   'Biology'   1289         1.38
6                   'Nursing'   1275         1.36
7                  'business'   1166         1.25
8                 'Education'   1162         1.24
9                   'nursing'    839         0.90
10              'Psychology '    821         0.88
11              'engineering'    773         0.83
12                'Economics'    730         0.78
13               'Accounting'    730         0.78
14        'civil engineering'    675         0.72
15                      'Law'    667         0.71
16                  'biology'    655         0.70
17                  'english'    649         0.69
18         'Computer Science'    643         0.69
19                  'History'    631         0.67


### Cross-check with `education` (1=<high school, 2=high school, 3=university degree, 4=graduate degree; 0 = no answer)

In [7]:
edu = pd.to_numeric(raw["education"], errors="coerce")
tab = pd.DataFrame({"n": edu.value_counts().sort_index(), "with major": edu[~blank].value_counts().sort_index()}).fillna(0).astype(int)
tab["% with major"] = (tab["with major"] / tab["n"] * 100).round(1)
tab.index = tab.index.map({0:"0 no answer",1:"1 <high school",2:"2 high school",3:"3 university degree",4:"4 graduate degree"})
print(tab.to_string())
uni = edu.isin([3, 4])
print(f"\nUniversity-level (3 or 4): {uni.sum():,} rows; major filled: {(uni & ~blank).sum():,} ({(uni & ~blank).sum()/uni.sum():.1%})")
print(f"Major filled but education below university: {(~uni & ~blank).sum():,} ({(~uni & ~blank).sum()/(~blank).sum():.1%} of all filled majors)")

                         n  with major  % with major
education                                           
0 no answer           1125         574          51.0
1 <high school       22665        5572          24.6
2 high school        62625       30227          48.3
3 university degree  39102       38066          97.4
4 graduate degree    20311       19187          94.5

University-level (3 or 4): 59,413 rows; major filled: 57,253 (96.4%)
Major filled but education below university: 36,373 (38.8% of all filled majors)


## 4. Data quality

In [8]:
items = groups[list(groups)[0]]
vals = raw[items].apply(pd.to_numeric, errors="coerce")
print("non-numeric cells in items:", int(vals.isna().sum().sum()))
print("value counts over all 48 items x all rows (codebook says 1-5, so 0 is undocumented):")
print(pd.Series(vals.values.ravel()).value_counts().sort_index().to_string())
zero_cells = (vals == 0)
print(f"\nrows with at least one 0 (no answer): {zero_cells.any(axis=1).sum():,} ({zero_cells.any(axis=1).mean():.1%})")
print("rows with all 48 items in 1-5:", ((vals >= 1) & (vals <= 5)).all(axis=1).sum())
print("out-of-range (<0 or >5):", int(((vals < 0) | (vals > 5)).sum().sum()))

non-numeric cells in items: 0
value counts over all 48 items x all rows (codebook says 1-5, so 0 is undocumented):
0      21435
1    2026975
2    1229897
3    1474708
4    1301851
5     944878

rows with at least one 0 (no answer): 10,064 (6.9%)
rows with all 48 items in 1-5: 135764
out-of-range (<0 or >5): 0


In [9]:
vcl = raw[["VCL6", "VCL9", "VCL12"]].apply(pd.to_numeric, errors="coerce")
print("claimed to know each fake word:", (vcl == 1).sum().to_dict())
fail_vcl = (vcl == 1).any(axis=1)
print(f"rows failing validity check (ticked at least one fake word): {fail_vcl.sum():,} ({fail_vcl.mean():.1%})")
print("ticked all three:", (vcl == 1).all(axis=1).sum())

claimed to know each fake word: {'VCL6': 12133, 'VCL9': 8510, 'VCL12': 17937}
rows failing validity check (ticked at least one fake word): 29,285 (20.1%)
ticked all three: 1973


In [10]:
age = pd.to_numeric(raw["age"], errors="coerce")
print("non-numeric age:", age.isna().sum(), "| min", age.min(), "| median", age.median(), "| max", age.max())
bins = [-1, 12, 17, 24, 34, 49, 79, 99, 10**9]
labels = ["<13 (codebook says dropped)", "13-17", "18-24", "25-34", "35-49", "50-79", "80-99", ">=100"]
dist = pd.cut(age, bins=bins, labels=labels).value_counts().sort_index()
print(pd.DataFrame({"n": dist, "%": (dist / len(age) * 100).round(1)}).to_string())
print("\nlargest age values:", sorted(age.dropna().unique())[-8:])
print("impossible/implausible: age<13:", (age < 13).sum(), "| age>100:", (age > 100).sum(), "| age>=80:", (age >= 80).sum())

non-numeric age: 0 | min 13 | median 21.0 | max 2147483647
                                 n     %
age                                     
<13 (codebook says dropped)      0   0.0
13-17                        37519  25.7
18-24                        52172  35.8
25-34                        26901  18.4
35-49                        19669  13.5
50-79                         9455   6.5
80-99                           15   0.0
>=100                           90   0.1

largest age values: [2015, 9001, 1000000, 9355680, 67989858, 69859078, 1000000000, 2147483647]
impossible/implausible: age<13: 0 | age>100: 92 | age>=80: 112


In [11]:
ctry = raw["country"].replace("", "(blank)")
top10 = ctry.value_counts().head(10)
print(pd.DataFrame({"n": top10, "%": (top10 / len(ctry) * 100).round(2)}).to_string())
african = {"DZ","AO","BJ","BW","BF","BI","CV","CM","CF","TD","KM","CG","CD","CI","DJ","EG","GQ","ER","SZ","ET","GA","GM","GH","GN","GW","KE","LS","LR","LY","MG","MW","ML","MR","MU","MA","MZ","NA","NE","NG","RW","ST","SN","SC","SL","SO","ZA","SS","SD","TZ","TG","TN","UG","ZM","ZW","RE","YT","EH","SH"}
print("\nKenya (KE):", (ctry == "KE").sum())
af = ctry[ctry.isin(african)].value_counts()
print("All African country codes:", af.sum(), f"({af.sum()/len(ctry):.2%})"); print(af.head(10).to_string())
print("\nEast Africa: KE", (ctry=="KE").sum(), "| TZ", (ctry=="TZ").sum(), "| UG", (ctry=="UG").sum(), "| ET", (ctry=="ET").sum(), "| RW", (ctry=="RW").sum())
print("unique country codes:", ctry.nunique(), "| 'NONE' codes:", (ctry=="NONE").sum())

             n      %
country              
US       80579  55.26
MY        7841   5.38
CA        7256   4.98
SG        5769   3.96
GB        5533   3.79
AU        5524   3.79
PH        4848   3.32
IN        2216   1.52
NONE      1846   1.27
NZ        1558   1.07

Kenya (KE): 731
All African country codes: 2046 (1.40%)
country
ZA    745
KE    731
EG    135
NG     86
GH     54
MA     53
UG     25
MU     25
SD     24
BW     23

East Africa: KE 731 | TZ 9 | UG 25 | ET 10 | RW 1
unique country codes: 189 | 'NONE' codes: 1846


## 5. Usable-sample estimate (funnel)
Keep adults, pass the validity check, have all 48 items answered (1-5), and have a non-empty major. Each row shows what remains after that filter is added.

In [12]:
ok_age   = (age >= 18) & (age <= 100)
ok_vcl   = ~fail_vcl
ok_items = ((vals >= 1) & (vals <= 5)).all(axis=1)
ok_major = ~blank
steps = [("all rows", pd.Series(True, index=raw.index)), ("+ adult (18-100)", ok_age), ("+ passed VCL check", ok_vcl),
         ("+ complete RIASEC items", ok_items), ("+ non-empty major", ok_major)]
keep = pd.Series(True, index=raw.index); rows = []
for name, f in steps:
    keep &= f; rows.append((name, int(keep.sum()), round(keep.mean() * 100, 1)))
print(pd.DataFrame(rows, columns=["filter", "rows left", "% of all"]).to_string(index=False))
final = keep.copy()
print("\nfinal usable rows:", final.sum())
print("  ...also excluding placeholder-like majors (NA/none/...):", (final & ~junk).sum())
print("  ...of which education is university-level (3/4):", (final & ~junk & uni).sum())
print("  ...of which Kenya:", (final & (ctry == "KE")).sum(), "| any African country:", (final & ctry.isin(african)).sum())
print("  ...distinct raw majors among final rows (strip+lowercase):", major[final].str.strip().str.lower().nunique())
print("  ...rows with shared/retaken network location (uniqueNetworkLocation=2):", (final & (raw["uniqueNetworkLocation"] == "2")).sum())

                 filter  rows left  % of all
               all rows     145828     100.0
       + adult (18-100)     108217      74.2
     + passed VCL check      85876      58.9
+ complete RIASEC items      80349      55.1
      + non-empty major      61399      42.1

final usable rows: 61399
  ...also excluding placeholder-like majors (NA/none/...): 60888
  ...of which education is university-level (3/4): 41990
  ...of which Kenya: 233 | any African country: 816
  ...distinct raw majors among final rows (strip+lowercase): 7903
  ...rows with shared/retaken network location (uniqueNetworkLocation=2): 12535
